# Évaluer le modèle LightGBM

## Question étudiée

Le modèle apprend-il à prévoir les variations de prix mieux que la règle naïve ? On utilise un **backtest chronologique** : pour chaque année testée (2022 à 2026), l’apprentissage ne peut utiliser que des données antérieures à cette année. Cela reproduit mieux une vraie prévision et évite de donner au modèle des informations du futur.

Le test principal entraîne et évalue sur le panel brut. Une seconde expérience entraîne sur le panel nettoyé des prix aberrants, mais l’évalue toujours sur le panel brut, afin de comparer les prévisions sur les mêmes observations.

## 1. Backtest principal sur les données brutes

La fonction `run_backtest` entraîne un modèle distinct pour chaque horizon. Le résumé compare MAE, MAPE et gain par rapport à la naïve. Un gain positif signifie que l’erreur du modèle est inférieure à celle de la référence.

In [1]:
import sys; sys.path.append("..")
import pandas as pd
from src.evaluation.baselines import load_panel
from src.models.lgbm import run_backtest

brut, propre = load_panel(brut=True), load_panel(brut=False)
res = run_backtest(brut, brut)          # entraîné sur brut, testé sur brut

def score(r):
    return pd.Series({"MAE_naïve": r.ae_naive.mean(), "MAE_modèle": r.ae_model.mean(),
                      "MAPE_naïve_%": 100 * r.ape_naive.mean(),
                      "MAPE_modèle_%": 100 * r.ape_model.mean(), "n": len(r)})

print(res.groupby("h").apply(score).round(1))                       # tous les mois
print(res[~res.bad].groupby("h").apply(score).round(1))             # sans mois figés
print(res[res.h == 1].groupby("year").apply(score).round(1))        # par année
print(res[res.h == 1].groupby("commodity").apply(score).round(1))   # par produit

   MAE_naïve  MAE_modèle  MAPE_naïve_%  MAPE_modèle_%        n
h                                                             
1       39.8        40.0          10.0           10.0  16291.0
2       57.6        55.1          14.5           13.9  16258.0
3       70.8        64.9          17.8           16.4  16251.0
   MAE_naïve  MAE_modèle  MAPE_naïve_%  MAPE_modèle_%        n
h                                                             
1       41.6        41.0          10.7           10.5  12789.0
2       60.8        56.8          15.8           14.9  11903.0
3       74.9        67.5          19.6           17.9  11611.0
      MAE_naïve  MAE_modèle  MAPE_naïve_%  MAPE_modèle_%       n
year                                                            
2022       38.5        37.3          10.8           10.5  3582.0
2023       31.8        32.2           7.4            7.4  3501.0
2024       32.0        35.4           7.1            7.8  3496.0
2025       56.1        54.2          12.7    

## 2. Sensibilité au nettoyage des valeurs aberrantes

Le modèle voit des données nettoyées pendant l’apprentissage, mais ses prédictions sont comparées aux prix bruts du test. Cette expérience indique si le retrait des valeurs aberrantes change sensiblement les résultats.

In [2]:
res_p = run_backtest(propre, brut)      # entraîné sur nettoyé, testé sur brut
print(res_p.groupby("h").apply(score).round(1))

   MAE_naïve  MAE_modèle  MAPE_naïve_%  MAPE_modèle_%        n
h                                                             
1       39.8        39.8          10.0            9.9  16291.0
2       57.6        55.2          14.5           13.9  16258.0
3       70.8        64.9          17.8           16.4  16251.0


## 3. Résultats par année et par produit à trois mois

Une moyenne générale peut cacher des différences. Cette cellule détaille le gain du modèle à trois mois selon l’année puis selon le produit, pour voir si l’amélioration est régulière ou concentrée dans certains cas.

In [3]:
def score(r):
    n, m = r.ape_naive.mean(), r.ape_model.mean()
    return pd.Series({"MAPE_naïve_%": 100*n, "MAPE_modèle_%": 100*m,
                      "gain_%": 100*(n-m)/n, "n": len(r)})

r3 = res[res.h == 3]
print(r3.groupby("year").apply(score).round(1))
print(r3.groupby("commodity").apply(score).round(1))

      MAPE_naïve_%  MAPE_modèle_%  gain_%       n
year                                             
2022          18.1           16.3    10.2  3573.0
2023          12.6           11.8     6.5  3495.0
2024          13.5           13.2     2.5  3495.0
2025          19.3           17.1    11.2  3488.0
2026          30.5           28.3     6.9  2200.0
                     MAPE_naïve_%  MAPE_modèle_%  gain_%       n
commodity                                                       
Beans (red)                  22.4           20.7     7.7  2008.0
Beans (white)                24.1           21.5    11.0  2043.0
Cassava meal (gari)          13.6           12.8     6.1  2037.0
Maize (white)                18.9           17.4     7.9  2024.0
Millet                       17.2           15.8     7.8  2004.0
Rice (local)                 11.2           10.4     6.4  2042.0
Sorghum (red)                16.9           15.7     7.0  2065.0
Soybeans                     18.6           17.4     6.9  2028.0
